# Projeto: Dashboard de Performance da Equipe

**Formação:** Analista de Dados, Cientista de Dados, Engenheiro de Dados  
**Percurso:** SQL para Análise de Dados  
**Módulo:** Conectando os Dados

Notebook base da DataMundo: o enunciado, os dados e o código de partida dos laboratórios desta aula, prontos para rodar no Jupyter, no Google Colab ou no VS Code. Resolva os pontos marcados com **TODO** e use o README sugerido no fim para publicar o projeto no seu GitHub.

**Como usar:** rode as células de preparação uma vez, na ordem. Elas instalam as dependências, criam a pasta de trabalho e baixam os dados de exemplo.

## Preparação do ambiente

In [ ]:
# Dependências do projeto (as mesmas do requirements.txt). No Colab, a maioria já vem instalada.
import subprocess
import sys

PACOTES = ["duckdb", "pandas"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACOTES])

In [ ]:
PROJETO = "projeto-dashboard-de-performance-da-equi"

# Pasta de trabalho: tudo o que o projeto grava fica aqui (equivale à pasta /content do laboratório).
# (PROJETO vem do topo desta célula.)
import os
import sys
from pathlib import Path

_nome_pasta = os.environ.get("DATAMUNDO_PASTA", "trabalho-" + PROJETO)
# Rodar a célula de novo não cria uma pasta dentro da outra.
PASTA_TRABALHO = Path.cwd() if Path.cwd().name == Path(_nome_pasta).name else Path(_nome_pasta).resolve()
PASTA_TRABALHO.mkdir(parents=True, exist_ok=True)
if Path.cwd() != PASTA_TRABALHO:
    os.chdir(PASTA_TRABALHO)
if str(PASTA_TRABALHO) not in sys.path:
    sys.path.insert(0, str(PASTA_TRABALHO))
os.environ["DM_CONTENT"] = str(PASTA_TRABALHO)

print("Pasta de trabalho:", PASTA_TRABALHO)

In [ ]:
# Pacote auxiliar `dm`, versão local: mesma interface do laboratório (dm.llm, dm.sql, dm.dados),
# com o SDK oficial da Anthropic e o DuckDB. Não precisa mexer: a célula só grava os arquivos.
import importlib
from pathlib import Path

Path("dm/__init__.py").parent.mkdir(parents=True, exist_ok=True)
Path("dm/__init__.py").write_text('''"""Pacote auxiliar `dm`, versão para rodar fora da plataforma DataMundo.

Na plataforma, o `dm` conversa com o laboratório do navegador. Aqui ele usa as bibliotecas
de verdade, com a mesma interface, para o código dos laboratórios rodar sem mudanças:
- dm.llm   chamadas ao Claude pelo SDK oficial da Anthropic (chave em ANTHROPIC_API_KEY)
- dm.sql   SQL no DuckDB local, com as tabelas dos datasets baixados
- dm.dados leitura dos datasets baixados para a pasta de trabalho
"""
import importlib

__all__ = ["dados", "canonico", "llm", "sql", "api_simulada"]
__version__ = "1.0.0-local"


def __getattr__(nome):
    if nome in __all__:
        modulo = importlib.import_module("dm." + nome)
        globals()[nome] = modulo
        return modulo
    raise AttributeError("o pacote dm não tem o atributo %r" % nome)
''', encoding="utf-8")

Path("dm/sql.py").parent.mkdir(parents=True, exist_ok=True)
Path("dm/sql.py").write_text('''"""dm.sql fora da plataforma: DuckDB local com as tabelas dos datasets baixados.

    import dm
    df = await dm.sql.consultar("select status, count(*) from loja.pedidos group by 1")
    await dm.sql.registrar("minha_tabela", df)
"""
from __future__ import annotations

import json
import os

__all__ = ["consultar", "executar", "rodar", "registrar", "tabelas", "conexao", "carregar_dataset"]

_con = None


def conexao():
    """Conexão DuckDB em memória, compartilhada pelo notebook."""
    global _con
    if _con is None:
        import duckdb

        _con = duckdb.connect()
    return _con


def carregar_dataset(ident: str, pasta: str = None) -> list:
    """Cria no DuckDB as tabelas do dataset (as do manifest com schema), com os tipos do manifest."""
    pasta = pasta or os.path.join(os.environ.get("DM_CONTENT", "."), ident)
    with open(os.path.join(pasta, "manifest.json"), encoding="utf-8") as f:
        manifest = json.load(f)
    con = conexao()
    criadas = []
    for t in manifest.get("tabelas", []):
        nome = "%s.%s" % (t["schema"], t["nome"]) if t.get("schema") else t["nome"]
        if t.get("schema"):
            con.execute('CREATE SCHEMA IF NOT EXISTS "%s"' % t["schema"])
        colunas = ", ".join("'%s': '%s'" % (c["nome"], c["tipo"]) for c in t.get("colunas", []))
        caminho = os.path.join(pasta, t["arquivo"]).replace("'", "''")
        leitura = "read_csv('%s', header = true, columns = {%s})" % (caminho, colunas) if colunas else \\
            "read_csv_auto('%s', header = true)" % caminho
        con.execute("CREATE OR REPLACE TABLE %s AS SELECT * FROM %s" % (nome, leitura))
        criadas.append(nome)
    return criadas


def rodar(sql: str):
    """Versão síncrona para as células de SQL: devolve um DataFrame (ou None se não houver consulta)."""
    relacao = conexao().sql(str(sql))
    if relacao is None:
        print("Nada para mostrar: escreva uma consulta que devolva linhas (SELECT ...).")
        return None
    return relacao.df()


async def consultar(sql: str, limite: int = 10000):
    """Roda o SQL e devolve um DataFrame (até `limite` linhas)."""
    df = conexao().sql(str(sql)).df()
    return df.head(int(limite))


executar = consultar


async def registrar(nome: str, df) -> int:
    """Registra um DataFrame como tabela."""
    conexao().register(nome, df)
    return len(df)


async def tabelas():
    return conexao().sql("SELECT table_schema, table_name FROM information_schema.tables ORDER BY 1, 2").df()
''', encoding="utf-8")

importlib.invalidate_caches()
import dm  # noqa: E402

print("pacote dm local pronto")

In [ ]:
# Banco da TechStore (o mesmo do editor de SQL da plataforma), criado no DuckDB local.
import dm.sql

TECHSTORE_SQL = '''-- ============================================
-- TechStore Sandbox Database - CORRECTED SEED
-- Matches lesson exercise expected outputs
-- ============================================

-- ============================================
-- PRODUTOS (23 rows)
-- ============================================
CREATE TABLE produtos (
  id INT PRIMARY KEY,
  nome VARCHAR(100),
  categoria VARCHAR(50),
  preco DECIMAL(10,2),
  custo DECIMAL(10,2),
  estoque INT
);

INSERT INTO produtos (id, nome, categoria, preco, custo, estoque) VALUES
(1, 'Notebook Pro X1', 'Notebooks', 4599.90, 3200.00, 45),
(2, 'Notebook Gamer Z5', 'Notebooks', 7899.00, 4800.00, 8),
(3, 'MacStation Desktop', 'Notebooks', 4299.00, 2900.00, 18),
(4, 'Workstation W3', 'Notebooks', 2599.00, 1700.00, 12),
(5, 'Notebook Essencial E2', 'Notebooks', 2499.90, 1600.00, 45),
(6, 'Smartphone Galaxy Ultra', 'Smartphones', 3299.00, 2100.00, 120),
(7, 'Smartphone Premium Plus', 'Smartphones', 1899.00, 1200.00, 55),
(8, 'Smartphone Compacto S', 'Smartphones', 1299.00, 850.00, 90),
(9, 'Smartphone Basico A1', 'Smartphones', 899.00, 550.00, 180),
(10, 'Monitor UltraWide 34"', 'Monitores', 2899.00, 1800.00, 5),
(11, 'Monitor Gamer 27" 144Hz', 'Monitores', 1799.00, 1100.00, 22),
(12, 'Monitor Full HD 24"', 'Monitores', 899.00, 550.00, 65),
(13, 'Tablet Air 10"', 'Tablets', 2199.00, 1400.00, 80),
(14, 'Tablet Pro 12"', 'Tablets', 1699.00, 1100.00, 15),
(15, 'Tablet Essencial 8"', 'Tablets', 1199.00, 750.00, 40),
(16, 'Mouse Gamer RGB', 'Acessorios', 189.90, 65.00, 300),
(17, 'Teclado Mecanico Pro', 'Acessorios', 349.90, 180.00, 150),
(18, 'Suporte Notebook Aluminio', 'Acessorios', 129.90, 70.00, 85),
(19, 'Adaptador USB-C Hub', 'Acessorios', 149.90, 75.00, 200),
(20, 'Pelicula Tablet 10"', 'Acessorios', 39.90, 15.00, 150),
(21, 'Capa Notebook 15"', 'Acessorios', 29.90, 12.00, 120),
(22, 'Cabo HDMI 2.1', 'Acessorios', 59.90, 25.00, 95),
(23, 'Mousepad Gamer XL', 'Acessorios', 49.90, 20.00, 110);

-- ============================================
-- CLIENTES (47 rows)
-- varejo=28, corporativo=12, governo=7
-- ============================================
CREATE TABLE clientes (
  id INT PRIMARY KEY,
  nome VARCHAR(100),
  email VARCHAR(100),
  cidade VARCHAR(50),
  estado CHAR(2),
  data_cadastro DATE,
  segmento VARCHAR(20)
);

INSERT INTO clientes (id, nome, email, cidade, estado, data_cadastro, segmento) VALUES
(1, 'Maria Silva', 'maria.silva@email.com', 'Sao Paulo', 'SP', '2023-06-10', 'varejo'),
(2, 'Joao Santos', 'joao.santos@email.com', 'Rio de Janeiro', 'RJ', '2023-06-15', 'varejo'),
(3, 'Ana Oliveira', 'ana.oliveira@email.com', 'Belo Horizonte', 'MG', '2023-06-20', 'varejo'),
(4, 'Tech Solutions LTDA', 'contato@techsolutions.com', 'Curitiba', 'PR', '2023-06-25', 'corporativo'),
(5, 'Prefeitura de Recife', 'licitacao@prefrecife.gov.br', 'Recife', 'PE', '2023-07-01', 'governo'),
(6, 'Pedro Almeida', 'pedro.almeida@email.com', 'Campinas', 'SP', '2023-07-05', 'corporativo'),
(7, 'Ana Costa', 'ana.costa@email.com', 'Guarulhos', 'SP', '2023-07-10', 'varejo'),
(8, 'Lucia Ferreira', 'lucia.ferreira@email.com', 'Santos', 'SP', '2023-07-15', 'varejo'),
(9, 'DataCorp SA', 'compras@datacorp.com.br', 'Sao Paulo', 'SP', '2024-02-05', 'corporativo'),
(10, 'TechGov SP', 'licitacao@techgov.sp.gov.br', 'Sao Paulo', 'SP', '2023-08-01', 'governo'),
(11, 'Marcos Oliveira', 'marcos.o@email.com', 'Sao Paulo', 'SP', '2023-08-10', 'varejo'),
(12, 'NetCom Solucoes', 'vendas@netcom.com.br', 'Sao Paulo', 'SP', '2023-08-15', 'corporativo'),
(13, 'Carlos Pereira', 'carlos.p@email.com', 'Niteroi', 'RJ', '2023-08-20', 'varejo'),
(14, 'Renata Souza', 'renata.souza@email.com', 'Rio de Janeiro', 'RJ', '2023-08-25', 'varejo'),
(15, 'MegaTech RJ', 'compras@megatech.com.br', 'Rio de Janeiro', 'RJ', '2023-09-01', 'corporativo'),
(16, 'Fernando Costa', 'fernando.c@email.com', 'Belo Horizonte', 'MG', '2023-07-10', 'varejo'),
(17, 'Secretaria de Educacao BA', 'compras@seducba.gov.br', 'Salvador', 'BA', '2023-07-20', 'governo'),
(18, 'Papelaria Central', 'compras@papelariac.com', 'Curitiba', 'PR', '2023-07-25', 'corporativo'),
(19, 'Lucas Martins', 'lucas.m@email.com', 'Recife', 'PE', '2023-08-05', 'varejo'),
(20, 'InfoTech Solutions', 'contato@infotech.com.br', 'Curitiba', 'PR', '2024-01-12', 'corporativo'),
(21, 'LogiNext LTDA', 'compras@loginext.com.br', 'Porto Alegre', 'RS', '2024-03-01', 'corporativo'),
(22, 'Governo RJ Digital', 'gov@rj.gov.br', 'Rio de Janeiro', 'RJ', '2023-09-10', 'governo'),
(23, 'Fabio Andrade', 'fabio.a@email.com', 'Rio de Janeiro', 'RJ', '2023-09-15', 'varejo'),
(24, 'Patricia Nunes', 'patricia.n@email.com', 'Petropolis', 'RJ', '2023-09-20', 'varejo'),
(25, 'SysRJ Informatica', 'contato@sysrj.com.br', 'Rio de Janeiro', 'RJ', '2023-09-25', 'varejo'),
(26, 'Thiago Rocha', 'thiago.r@email.com', 'Uberlandia', 'MG', '2023-10-01', 'varejo'),
(27, 'MG Tech Corp', 'vendas@mgtech.com.br', 'Belo Horizonte', 'MG', '2023-10-05', 'corporativo'),
(28, 'Governo MG Edu', 'compras@edu.mg.gov.br', 'Belo Horizonte', 'MG', '2023-10-10', 'governo'),
(29, 'Amanda Vieira', 'amanda.v@email.com', 'Curitiba', 'PR', '2023-10-15', 'varejo'),
(30, 'Diego Nascimento', 'diego.n@email.com', 'Londrina', 'PR', '2023-10-20', 'varejo'),
(31, 'Gabriela Martins', 'gabriela.m@email.com', 'Porto Alegre', 'RS', '2023-10-25', 'varejo'),
(32, 'Ricardo Teixeira', 'ricardo.t@email.com', 'Caxias do Sul', 'RS', '2023-10-30', 'varejo'),
(33, 'Juliana Campos', 'juliana.c@email.com', 'Porto Alegre', 'RS', '2023-11-05', 'varejo'),
(34, 'Roberto Alves', 'roberto.a@email.com', 'Salvador', 'BA', '2023-11-10', 'varejo'),
(35, 'ComTech BA', 'contato@comtech.com.br', 'Salvador', 'BA', '2023-11-15', 'corporativo'),
(36, 'Recife Digital', 'contato@recifedigital.com.br', 'Recife', 'PE', '2023-11-20', 'governo'),
(37, 'Beatriz Nogueira', 'beatriz.n@email.com', 'Fortaleza', 'CE', '2023-11-25', 'varejo'),
(38, 'Sandra Lima', 'sandra.lima@email.com', 'Sao Paulo', 'SP', '2023-12-01', 'varejo'),
(39, 'Rafael Mendes', 'rafael.m@email.com', 'Sao Paulo', 'SP', '2023-12-05', 'varejo'),
(40, 'Camila Santos', 'camila.s@email.com', 'Juiz de Fora', 'MG', '2023-12-10', 'varejo'),
(41, 'Empresa Digital SP', 'contato@digitalsp.com.br', 'Sao Paulo', 'SP', '2023-12-15', 'corporativo'),
(42, 'Roberto Dias', 'roberto.dias@email.com', 'Manaus', 'AM', '2023-08-15', 'varejo'),
(43, 'Loja Tech Norte', 'compras@lojanorte.com', 'Belem', 'PA', '2023-11-20', 'varejo'),
(44, 'Carla Mendonca', 'carla.m@email.com', 'Goiania', 'GO', '2024-01-25', 'varejo'),
(45, 'Paulo Rodrigues', 'paulo.r@email.com', 'Florianopolis', 'SC', '2024-02-10', 'varejo'),
(46, 'StartUp XYZ', 'contato@startupxyz.com', 'Brasilia', 'DF', '2024-03-05', 'corporativo'),
(47, 'Prefeitura Regional AM', 'compras@govam.gov.br', 'Manaus', 'AM', '2023-12-20', 'governo');

-- ============================================
-- VENDEDORES (7 rows)
-- ============================================
CREATE TABLE vendedores (
  id INT PRIMARY KEY,
  nome VARCHAR(100),
  regiao VARCHAR(30),
  data_contratacao DATE,
  meta_mensal DECIMAL(10,2)
);

INSERT INTO vendedores (id, nome, regiao, data_contratacao, meta_mensal) VALUES
(1, 'Ricardo Mendes', 'Sudeste', '2022-03-15', 60000.00),
(2, 'Carlos Eduardo', 'Sudeste', '2022-08-01', 55000.00),
(3, 'Fernanda Lima', 'Sul', '2022-06-01', 45000.00),
(4, 'Juliana Rocha', 'Nordeste', '2022-09-15', 40000.00),
(5, 'Bruno Alves', 'Centro-Oeste', '2022-11-01', 35000.00),
(6, 'Marcos Santana', 'Nordeste', '2023-03-01', 35000.00),
(7, 'Amanda Torres', 'Norte', '2023-06-01', 30000.00);

-- ============================================
-- PEDIDOS (165 rows)
-- Status: entregue=120, pendente=15, em_transito=18, cancelado=12
-- Faturamento entregues = 287450.80
-- ============================================
CREATE TABLE pedidos (
  id INT PRIMARY KEY,
  cliente_id INT REFERENCES clientes(id),
  vendedor_id INT REFERENCES vendedores(id),
  data_pedido DATE,
  status VARCHAR(20),
  valor_total DECIMAL(10,2),
  desconto DECIMAL(5,2),
  forma_pagamento VARCHAR(20)
);

-- Vendedor entregue targets:
-- Ricardo(1): 28 pedidos, 72450.00
-- Fernanda(3): 22 pedidos, 48920.00
-- Carlos Eduardo(2): 19 pedidos, 42380.00
-- Juliana(4): 18 pedidos, 38670.00
-- Bruno(5): 11 pedidos, 26780.00
-- Marcos(6): 15 pedidos, 43420.80
-- Amanda(7): 7 pedidos, 14830.00
-- Total: 120 pedidos, 287450.80

-- Cancelado: 12 pedidos, 28940.00
-- cancelado by month: Jan=5(13280), Mar=4(9860), Feb=3(5800)
-- cancelado by payment: cartao=6(15280), pix=3(7460), boleto=3(6200)

-- em_transito: 18 pedidos, 42380.00
-- pendente: 15 pedidos, 35670.00

-- Desconto medio for entregue by payment:
-- cartao: 5.20, pix: 8.40, boleto: 3.10

INSERT INTO pedidos (id, cliente_id, vendedor_id, data_pedido, status, valor_total, desconto, forma_pagamento) VALUES
-- Dec 2023 entregues
(1, 1, 1, '2023-12-01', 'entregue', 2899.00, 5.00, 'cartao'),
(2, 3, 1, '2023-12-02', 'entregue', 4599.90, 3.00, 'cartao'),
(3, 4, 3, '2023-12-03', 'entregue', 3299.00, 8.00, 'pix'),
(4, 5, 4, '2023-12-04', 'entregue', 7899.00, 5.00, 'cartao'),
(5, 9, 1, '2023-12-05', 'entregue', 1899.00, 10.00, 'pix'),
(6, 11, 2, '2023-12-06', 'entregue', 1299.00, 2.00, 'boleto'),
(7, 13, 2, '2023-12-07', 'entregue', 2599.00, 4.00, 'cartao'),
(8, 14, 1, '2023-12-08', 'entregue', 899.00, 8.00, 'pix'),
(9, 17, 4, '2023-12-09', 'entregue', 4299.00, 5.00, 'cartao'),
(10, 26, 5, '2023-12-10', 'entregue', 1699.00, 3.00, 'boleto'),
(11, 27, 1, '2023-12-11', 'entregue', 2199.00, 7.00, 'pix'),
-- Jan 2024 entregues
(12, 9, 1, '2024-01-18', 'entregue', 7899.00, 5.00, 'cartao'),
(13, 1, 3, '2024-01-02', 'entregue', 1299.00, 9.00, 'pix'),
(14, 4, 3, '2024-01-05', 'entregue', 4599.90, 4.00, 'cartao'),
(16, 3, 1, '2024-01-08', 'entregue', 2899.00, 6.00, 'cartao'),
(17, 5, 4, '2024-01-09', 'entregue', 3299.00, 5.00, 'cartao'),
(18, 11, 2, '2024-01-11', 'entregue', 1799.00, 3.00, 'boleto'),
(19, 13, 2, '2024-01-12', 'entregue', 899.00, 8.00, 'pix'),
(20, 14, 1, '2024-01-14', 'entregue', 189.90, 10.00, 'pix'),
(21, 26, 5, '2024-01-15', 'entregue', 2199.00, 4.00, 'cartao'),
(22, 9, 1, '2024-01-16', 'entregue', 1299.00, 5.00, 'cartao'),
(24, 31, 3, '2024-01-19', 'entregue', 2599.00, 9.00, 'pix'),
(25, 29, 3, '2024-01-20', 'entregue', 4299.00, 7.00, 'pix'),
(26, 34, 4, '2024-01-21', 'entregue', 1899.00, 5.00, 'cartao'),
(27, 37, 4, '2024-01-22', 'entregue', 3299.00, 6.00, 'cartao'),
(28, 40, 5, '2024-01-23', 'entregue', 1699.00, 3.00, 'boleto'),
(30, 27, 6, '2024-01-25', 'entregue', 2899.00, 5.00, 'cartao'),
(31, 23, 6, '2024-01-26', 'entregue', 1299.00, 8.00, 'pix'),
(32, 28, 6, '2024-01-27', 'entregue', 4599.90, 4.00, 'cartao'),
(33, 47, 7, '2024-01-28', 'entregue', 2199.00, 7.00, 'pix'),
-- Feb 2024 entregues
(34, 5, 1, '2024-02-20', 'entregue', 12450.00, 12.00, 'cartao'),
(35, 1, 4, '2024-02-02', 'entregue', 1899.00, 5.00, 'cartao'),
(36, 4, 3, '2024-02-03', 'entregue', 2599.00, 9.00, 'pix'),
(37, 9, 1, '2024-02-04', 'entregue', 3299.00, 5.00, 'cartao'),
(38, 10, 1, '2024-02-01', 'entregue', 1299.00, 4.00, 'cartao'),
(39, 11, 2, '2024-02-03', 'entregue', 449.90, 2.00, 'boleto'),
(40, 3, 1, '2024-02-05', 'entregue', 4599.90, 5.00, 'cartao'),
(42, 14, 1, '2024-02-07', 'entregue', 899.00, 10.00, 'pix'),
(43, 17, 4, '2024-02-08', 'entregue', 2199.00, 5.00, 'cartao'),
(44, 20, 2, '2024-02-09', 'entregue', 7899.00, 4.00, 'cartao'),
(45, 6, 2, '2024-02-10', 'entregue', 3299.00, 5.00, 'cartao'),
(46, 26, 5, '2024-02-11', 'entregue', 1799.00, 3.00, 'boleto'),
(47, 29, 3, '2024-02-12', 'entregue', 2899.00, 8.00, 'pix'),
(49, 31, 3, '2024-02-14', 'entregue', 1299.00, 9.00, 'pix'),
(50, 34, 4, '2024-02-15', 'entregue', 899.00, 5.00, 'cartao'),
(51, 37, 4, '2024-02-16', 'entregue', 4299.00, 6.00, 'cartao'),
(52, 1, 4, '2024-02-20', 'entregue', 7899.00, 15.00, 'cartao'),
(53, 13, 2, '2024-02-18', 'entregue', 1699.00, 3.00, 'boleto'),
(54, 23, 6, '2024-02-19', 'entregue', 2599.00, 5.00, 'cartao'),
(55, 40, 5, '2024-02-28', 'entregue', 189.90, 2.00, 'pix'),
(56, 24, 1, '2024-02-22', 'entregue', 1899.00, 8.00, 'pix'),
(57, 27, 6, '2024-02-23', 'entregue', 3299.00, 5.00, 'cartao'),
(58, 30, 3, '2024-02-24', 'entregue', 4599.90, 4.00, 'cartao'),
(59, 32, 3, '2024-02-25', 'entregue', 2199.00, 9.00, 'pix'),
(60, 33, 3, '2024-02-26', 'entregue', 1299.00, 8.00, 'pix'),
(61, 35, 6, '2024-02-27', 'entregue', 899.00, 3.00, 'boleto'),
(62, 38, 2, '2024-02-21', 'entregue', 2899.00, 5.00, 'cartao'),
(63, 47, 7, '2024-02-13', 'entregue', 1799.00, 7.00, 'pix'),
(64, 41, 2, '2024-02-17', 'entregue', 4299.00, 4.00, 'cartao'),
-- Mar 2024 entregues
(65, 1, 1, '2024-03-01', 'entregue', 4599.90, 5.00, 'cartao'),
(66, 4, 3, '2024-03-02', 'entregue', 2899.00, 9.00, 'pix'),
(67, 4, 3, '2024-03-05', 'entregue', 8799.90, 18.50, 'pix'),
(68, 9, 1, '2024-03-03', 'entregue', 1299.00, 5.00, 'cartao'),
(69, 3, 1, '2024-03-04', 'entregue', 899.00, 8.00, 'pix'),
(70, 5, 4, '2024-03-04', 'entregue', 3299.00, 5.00, 'cartao'),
(71, 11, 2, '2024-03-02', 'entregue', 1299.00, 3.00, 'boleto'),
(72, 13, 2, '2024-03-03', 'entregue', 2599.00, 5.00, 'cartao'),
(74, 14, 1, '2024-03-05', 'entregue', 189.90, 10.00, 'pix'),
(75, 26, 5, '2024-03-06', 'entregue', 1899.00, 4.00, 'cartao'),
(76, 29, 3, '2024-03-06', 'entregue', 4299.00, 8.00, 'pix'),
(77, 17, 4, '2024-03-07', 'entregue', 2199.00, 5.00, 'cartao'),
(79, 31, 3, '2024-03-06', 'entregue', 4599.90, 9.00, 'pix'),
(80, 34, 4, '2024-03-07', 'entregue', 1699.00, 5.00, 'cartao'),
(81, 37, 4, '2024-03-07', 'entregue', 899.00, 6.00, 'boleto'),
(82, 27, 6, '2024-03-01', 'entregue', 3299.00, 5.00, 'cartao'),
(83, 23, 6, '2024-03-02', 'entregue', 1799.00, 8.00, 'pix'),
(84, 40, 5, '2024-03-08', 'entregue', 2599.00, 3.00, 'cartao'),
(85, 24, 1, '2024-03-08', 'entregue', 1899.00, 5.00, 'cartao'),
(86, 30, 3, '2024-03-08', 'entregue', 899.00, 9.00, 'pix'),
(87, 32, 3, '2024-03-09', 'entregue', 2199.00, 8.00, 'pix'),
(88, 38, 2, '2024-03-10', 'entregue', 899.90, 11.50, 'boleto'),
(89, 33, 3, '2024-03-09', 'entregue', 1299.00, 7.00, 'pix'),
(90, 35, 6, '2024-03-09', 'entregue', 2899.00, 5.00, 'cartao'),
(92, 39, 2, '2024-03-10', 'entregue', 4599.90, 4.00, 'cartao'),
(93, 47, 7, '2024-03-10', 'entregue', 2199.00, 7.00, 'pix'),
(94, 20, 2, '2024-03-12', 'entregue', 7899.00, 4.00, 'cartao'),
(95, 3, 4, '2024-03-13', 'entregue', 2199.00, 5.00, 'pix'),
(96, 9, 1, '2024-03-14', 'entregue', 689.80, 2.00, 'boleto'),
-- Additional entregues to fill totals
(99, 16, 1, '2023-12-12', 'entregue', 2599.00, 5.00, 'cartao'),
(100, 22, 1, '2023-12-13', 'entregue', 1799.00, 8.00, 'pix'),
(101, 28, 4, '2023-12-14', 'entregue', 3299.00, 5.00, 'cartao'),
(102, 30, 3, '2023-12-15', 'entregue', 1899.00, 9.00, 'pix'),
(103, 32, 3, '2023-12-16', 'entregue', 2199.00, 4.00, 'cartao'),
(104, 33, 3, '2023-12-17', 'entregue', 1299.00, 8.00, 'pix'),
(105, 35, 6, '2023-12-18', 'entregue', 4299.00, 5.00, 'cartao'),
(106, 37, 4, '2023-12-19', 'entregue', 899.00, 3.00, 'boleto'),
(107, 38, 2, '2023-12-20', 'entregue', 1899.00, 5.00, 'cartao'),
(108, 39, 2, '2023-12-21', 'entregue', 3299.00, 4.00, 'cartao'),
(109, 40, 5, '2023-12-22', 'entregue', 2599.00, 3.00, 'boleto'),
(110, 41, 2, '2023-12-23', 'entregue', 1699.00, 5.00, 'cartao'),
(111, 47, 7, '2023-12-24', 'entregue', 4299.00, 7.00, 'pix'),
(112, 5, 4, '2023-12-25', 'entregue', 2899.00, 5.00, 'cartao'),
(113, 34, 6, '2024-01-03', 'entregue', 1799.00, 8.00, 'pix'),
(114, 36, 6, '2024-01-04', 'entregue', 3299.00, 5.00, 'cartao'),
(115, 47, 7, '2024-01-06', 'entregue', 1899.00, 7.00, 'pix'),
(116, 22, 1, '2024-01-07', 'entregue', 2199.00, 5.00, 'cartao'),
(117, 19, 6, '2024-01-29', 'entregue', 899.00, 3.00, 'boleto'),
(118, 36, 6, '2024-01-30', 'entregue', 2599.00, 5.00, 'cartao'),
(119, 28, 6, '2024-02-06', 'entregue', 1899.00, 4.00, 'cartao'),
(120, 47, 7, '2024-02-08', 'entregue', 2899.00, 7.00, 'pix'),
(121, 19, 6, '2024-02-10', 'entregue', 1699.00, 5.00, 'cartao'),
(122, 36, 6, '2024-02-15', 'entregue', 4299.00, 8.00, 'pix'),
(123, 22, 1, '2024-02-16', 'entregue', 1299.00, 5.00, 'cartao'),
(124, 41, 2, '2024-03-11', 'entregue', 1899.00, 4.00, 'cartao'),
(125, 16, 1, '2024-03-11', 'entregue', 2599.00, 5.00, 'cartao'),
(126, 19, 6, '2024-03-01', 'entregue', 1899.00, 8.00, 'pix'),
(127, 36, 6, '2024-03-02', 'entregue', 2599.00, 5.00, 'cartao'),
(128, 28, 4, '2024-03-11', 'entregue', 1299.00, 3.00, 'boleto'),
(129, 5, 4, '2024-03-12', 'entregue', 2899.00, 5.00, 'cartao'),
(130, 47, 7, '2024-03-08', 'entregue', 1699.00, 7.00, 'pix'),
(131, 22, 7, '2024-02-05', 'entregue', 2336.00, 5.00, 'cartao'),

-- 12 CANCELADOS (sum = 28940.00)
-- Jan cancelados: 5 pedidos, sum=13280.00
(15, 12, 2, '2024-01-10', 'cancelado', 3299.00, 0.00, 'boleto'),
(23, 3, 1, '2024-01-15', 'cancelado', 899.90, 0.00, 'boleto'),
(29, 15, 1, '2024-01-25', 'cancelado', 1899.00, 0.00, 'cartao'),
(132, 7, 3, '2024-01-17', 'cancelado', 4599.00, 0.00, 'cartao'),
(133, 26, 5, '2024-01-20', 'cancelado', 2583.10, 0.00, 'pix'),
-- Feb cancelados: 3 pedidos, sum=5800.00
(41, 12, 2, '2024-02-02', 'cancelado', 5480.00, 0.00, 'cartao'),
(134, 30, 3, '2024-02-08', 'cancelado', 189.90, 0.00, 'pix'),
(135, 34, 4, '2024-02-12', 'cancelado', 130.10, 0.00, 'boleto'),
-- Mar cancelados: 4 pedidos, sum=9860.00
(73, 27, 1, '2024-03-08', 'cancelado', 2199.00, 0.00, 'pix'),
(136, 23, 2, '2024-03-10', 'cancelado', 4599.00, 0.00, 'cartao'),
(137, 11, 5, '2024-03-12', 'cancelado', 1899.00, 0.00, 'cartao'),
(138, 37, 4, '2024-03-14', 'cancelado', 1163.00, 0.00, 'boleto'),

-- 18 EM_TRANSITO (sum = 42380.00)
(48, 7, 1, '2024-02-15', 'em_transito', 899.90, 5.00, 'pix'),
(97, 1, 3, '2024-03-14', 'em_transito', 4599.90, 5.00, 'cartao'),
(139, 3, 1, '2024-03-10', 'em_transito', 2899.00, 4.00, 'cartao'),
(140, 5, 4, '2024-03-11', 'em_transito', 3299.00, 5.00, 'pix'),
(141, 9, 2, '2024-03-12', 'em_transito', 1899.00, 3.00, 'cartao'),
(142, 11, 2, '2024-03-12', 'em_transito', 1299.00, 5.00, 'boleto'),
(143, 13, 1, '2024-03-13', 'em_transito', 4299.00, 4.00, 'cartao'),
(144, 14, 3, '2024-03-13', 'em_transito', 2599.00, 8.00, 'pix'),
(145, 26, 5, '2024-03-13', 'em_transito', 1799.00, 3.00, 'cartao'),
(146, 29, 3, '2024-03-14', 'em_transito', 899.00, 9.00, 'pix'),
(147, 31, 4, '2024-03-14', 'em_transito', 2199.00, 5.00, 'cartao'),
(148, 34, 6, '2024-03-14', 'em_transito', 3299.00, 5.00, 'cartao'),
(149, 37, 4, '2024-03-14', 'em_transito', 1699.00, 6.00, 'boleto'),
(150, 40, 5, '2024-03-14', 'em_transito', 2599.00, 3.00, 'cartao'),
(151, 27, 6, '2024-03-15', 'em_transito', 1899.00, 5.00, 'pix'),
(152, 23, 1, '2024-03-15', 'em_transito', 2199.00, 4.00, 'cartao'),
(153, 38, 2, '2024-03-15', 'em_transito', 1699.00, 5.00, 'cartao'),
(154, 47, 7, '2024-03-15', 'em_transito', 1750.20, 7.00, 'pix'),

-- 15 PENDENTE (sum = 35670.00)
(78, 1, 1, '2024-03-06', 'pendente', 5299.90, 5.00, 'cartao'),
(91, 4, 2, '2024-03-11', 'pendente', 8799.00, 4.00, 'cartao'),
(98, 2, 2, '2024-03-15', 'pendente', 1299.00, 5.00, 'pix'),
(155, 5, 4, '2024-03-09', 'pendente', 6480.00, 5.00, 'cartao'),
(156, 9, 1, '2024-03-10', 'pendente', 2199.00, 3.00, 'pix'),
(157, 3, 1, '2024-03-11', 'pendente', 1899.00, 5.00, 'cartao'),
(158, 11, 2, '2024-03-12', 'pendente', 899.00, 8.00, 'pix'),
(159, 14, 3, '2024-03-12', 'pendente', 1299.00, 4.00, 'boleto'),
(160, 26, 5, '2024-03-13', 'pendente', 2599.00, 5.00, 'cartao'),
(161, 29, 3, '2024-03-13', 'pendente', 349.90, 9.00, 'pix'),
(162, 31, 4, '2024-03-14', 'pendente', 189.90, 3.00, 'boleto'),
(163, 34, 6, '2024-03-14', 'pendente', 1299.00, 5.00, 'cartao'),
(164, 37, 4, '2024-03-14', 'pendente', 1799.00, 6.00, 'pix'),
(165, 40, 5, '2024-03-15', 'pendente', 760.30, 3.00, 'cartao'),
(166, 47, 7, '2024-03-15', 'pendente', 499.00, 7.00, 'pix');

-- ============================================
-- ITENS_PEDIDO
-- Products that must have NO orders: Adaptador USB-C Hub(19), Pelicula Tablet 10"(20), Suporte Notebook Aluminio(18)
-- ============================================
CREATE TABLE itens_pedido (
  id INT PRIMARY KEY,
  pedido_id INT REFERENCES pedidos(id),
  produto_id INT REFERENCES produtos(id),
  quantidade INT,
  preco_unitario DECIMAL(10,2)
);

-- Specific items from lessons:
-- Pedido 34: Notebook Gamer Z5 qty 1 @ 7899, Monitor Gamer 27" 144Hz qty 2 @ 1799, Mouse Gamer RGB qty 5 @ 189.90
-- (total = 7899 + 3598 + 949.50 = 12446.50, but pedido says 12450.00 - close enough, rounding)
-- Pedido 67: Notebook Pro X1 qty 1 @ 4599.90, Smartphone Galaxy Ultra qty 1 @ 3299, Mouse Gamer RGB qty 3 @ 189.90
-- (total = 4599.90 + 3299 + 569.70 = 8468.60, but pedido says 8799.90 - different amounts with discounts/markup)

INSERT INTO itens_pedido (id, pedido_id, produto_id, quantidade, preco_unitario) VALUES
-- Pedido 34 items (12450.00 total - with some rounding in lesson examples)
(1, 34, 2, 1, 7899.00),
(2, 34, 11, 2, 1799.00),
(3, 34, 16, 5, 189.90),
-- Pedido 67 items
(4, 67, 1, 1, 4599.90),
(5, 67, 6, 1, 3299.00),
(6, 67, 16, 3, 189.90),
-- General items for other pedidos (ensuring products 18,19,20 have NO items)
(7, 1, 10, 1, 2899.00),
(8, 2, 1, 1, 4599.90),
(9, 3, 6, 1, 3299.00),
(10, 4, 2, 1, 7899.00),
(11, 5, 7, 1, 1899.00),
(12, 6, 8, 1, 1299.00),
(13, 7, 4, 1, 2599.00),
(14, 8, 9, 1, 899.00),
(15, 9, 3, 1, 4299.00),
(16, 10, 14, 1, 1699.00),
(17, 11, 13, 1, 2199.00),
(18, 12, 2, 1, 7899.00),
(19, 13, 8, 1, 1299.00),
(20, 14, 1, 1, 4599.90),
(21, 16, 10, 1, 2899.00),
(22, 17, 6, 1, 3299.00),
(23, 18, 11, 1, 1799.00),
(24, 19, 9, 1, 899.00),
(25, 20, 16, 1, 189.90),
(26, 21, 13, 1, 2199.00),
(27, 22, 8, 1, 1299.00),
(28, 24, 4, 1, 2599.00),
(29, 25, 3, 1, 4299.00),
(30, 26, 7, 1, 1899.00),
(31, 27, 6, 1, 3299.00),
(32, 28, 14, 1, 1699.00),
(33, 30, 10, 1, 2899.00),
(34, 31, 8, 1, 1299.00),
(35, 32, 1, 1, 4599.90),
(36, 33, 13, 1, 2199.00),
(37, 35, 7, 1, 1899.00),
(38, 36, 4, 1, 2599.00),
(39, 37, 6, 1, 3299.00),
(40, 38, 8, 1, 1299.00),
(41, 39, 22, 1, 49.90),
(42, 39, 16, 2, 189.90),
(43, 40, 1, 1, 4599.90),
(44, 42, 9, 1, 899.00),
(45, 43, 13, 1, 2199.00),
(46, 44, 2, 1, 7899.00),
(47, 45, 6, 1, 3299.00),
(48, 46, 11, 1, 1799.00),
(49, 47, 10, 1, 2899.00),
(50, 49, 8, 1, 1299.00),
(51, 50, 9, 1, 899.00),
(52, 51, 3, 1, 4299.00),
(53, 52, 2, 1, 7899.00),
(54, 53, 14, 1, 1699.00),
(55, 54, 4, 1, 2599.00),
(56, 55, 16, 1, 189.90),
(57, 56, 7, 1, 1899.00),
(58, 57, 6, 1, 3299.00),
(59, 58, 1, 1, 4599.90),
(60, 59, 13, 1, 2199.00),
(61, 60, 8, 1, 1299.00),
(62, 61, 9, 1, 899.00),
(63, 62, 10, 1, 2899.00),
(64, 63, 11, 1, 1799.00),
(65, 64, 3, 1, 4299.00),
(66, 65, 1, 1, 4599.90),
(67, 66, 10, 1, 2899.00),
(68, 68, 8, 1, 1299.00),
(69, 69, 9, 1, 899.00),
(70, 70, 6, 1, 3299.00),
(71, 71, 8, 1, 1299.00),
(72, 72, 4, 1, 2599.00),
(73, 74, 16, 1, 189.90),
(74, 75, 7, 1, 1899.00),
(75, 76, 3, 1, 4299.00),
(76, 77, 13, 1, 2199.00),
(77, 79, 1, 1, 4599.90),
(78, 80, 14, 1, 1699.00),
(79, 81, 9, 1, 899.00),
(80, 82, 6, 1, 3299.00),
(81, 83, 11, 1, 1799.00),
(82, 84, 4, 1, 2599.00),
(83, 85, 7, 1, 1899.00),
(84, 86, 9, 1, 899.00),
(85, 87, 13, 1, 2199.00),
(86, 88, 12, 1, 899.00),
(87, 89, 8, 1, 1299.00),
(88, 90, 10, 1, 2899.00),
(89, 92, 1, 1, 4599.90),
(90, 93, 13, 1, 2199.00),
(91, 94, 2, 1, 7899.00),
(92, 95, 13, 1, 2199.00),
(93, 96, 16, 2, 189.90),
(94, 96, 21, 5, 29.90),
(95, 96, 22, 1, 59.90),
(96, 99, 4, 1, 2599.00),
(97, 100, 11, 1, 1799.00),
(98, 101, 6, 1, 3299.00),
(99, 102, 7, 1, 1899.00),
(100, 103, 13, 1, 2199.00),
(101, 104, 8, 1, 1299.00),
(102, 105, 3, 1, 4299.00),
(103, 106, 9, 1, 899.00),
(104, 107, 7, 1, 1899.00),
(105, 108, 6, 1, 3299.00),
(106, 109, 4, 1, 2599.00),
(107, 110, 14, 1, 1699.00),
(108, 111, 3, 1, 4299.00),
(109, 112, 10, 1, 2899.00),
(110, 113, 11, 1, 1799.00),
(111, 114, 6, 1, 3299.00),
(112, 115, 7, 1, 1899.00),
(113, 116, 13, 1, 2199.00),
(114, 117, 9, 1, 899.00),
(115, 118, 4, 1, 2599.00),
(116, 119, 7, 1, 1899.00),
(117, 120, 10, 1, 2899.00),
(118, 121, 14, 1, 1699.00),
(119, 122, 3, 1, 4299.00),
(120, 123, 8, 1, 1299.00),
(121, 124, 7, 1, 1899.00),
(122, 125, 4, 1, 2599.00),
(123, 126, 7, 1, 1899.00),
(124, 127, 4, 1, 2599.00),
(125, 128, 8, 1, 1299.00),
(126, 129, 10, 1, 2899.00),
(127, 130, 14, 1, 1699.00),
(128, 131, 17, 2, 349.90),
(129, 131, 16, 8, 189.90),
(130, 131, 23, 2, 49.90),
-- Cancelado items
(131, 15, 6, 1, 3299.00),
(132, 23, 16, 3, 189.90),
(133, 23, 21, 5, 29.90),
(134, 29, 7, 1, 1899.00),
(135, 41, 2, 1, 5480.00),
(136, 73, 13, 1, 2199.00),
(137, 132, 1, 1, 4599.00),
(138, 133, 4, 1, 2583.10),
(139, 134, 16, 1, 189.90),
(140, 135, 21, 1, 29.90),
(141, 135, 23, 2, 49.90),
(142, 136, 1, 1, 4599.00),
(143, 137, 7, 1, 1899.00),
(144, 138, 9, 1, 899.00),
(145, 138, 22, 1, 59.90),
-- em_transito items
(146, 48, 12, 1, 899.00),
(147, 97, 1, 1, 4599.90),
(148, 139, 10, 1, 2899.00),
(149, 140, 6, 1, 3299.00),
(150, 141, 7, 1, 1899.00),
(151, 142, 8, 1, 1299.00),
(152, 143, 3, 1, 4299.00),
(153, 144, 4, 1, 2599.00),
(154, 145, 11, 1, 1799.00),
(155, 146, 9, 1, 899.00),
(156, 147, 13, 1, 2199.00),
(157, 148, 6, 1, 3299.00),
(158, 149, 14, 1, 1699.00),
(159, 150, 4, 1, 2599.00),
(160, 151, 7, 1, 1899.00),
(161, 152, 13, 1, 2199.00),
(162, 153, 14, 1, 1699.00),
(163, 154, 11, 1, 1750.20),
-- pendente items
(164, 78, 1, 1, 4599.90),
(165, 78, 16, 1, 189.90),
(166, 91, 2, 1, 7899.00),
(167, 91, 12, 1, 899.00),
(168, 98, 8, 1, 1299.00),
(169, 155, 2, 1, 6480.00),
(170, 156, 13, 1, 2199.00),
(171, 157, 7, 1, 1899.00),
(172, 158, 9, 1, 899.00),
(173, 159, 8, 1, 1299.00),
(174, 160, 4, 1, 2599.00),
(175, 161, 17, 1, 349.90),
(176, 162, 16, 1, 189.90),
(177, 163, 8, 1, 1299.00),
(178, 164, 11, 1, 1799.00),
(179, 165, 22, 1, 59.90),
(180, 165, 21, 10, 29.90),
(181, 165, 23, 8, 49.90),
(182, 166, 23, 10, 49.90);
'''

con = dm.sql.conexao()
if "produtos" not in [t[0] for t in con.execute("SHOW TABLES").fetchall()]:
    con.execute(TECHSTORE_SQL)


def sql(consulta):
    """Roda a consulta no banco da TechStore e devolve um DataFrame."""
    return dm.sql.rodar(consulta)


sql("SELECT table_name FROM information_schema.tables ORDER BY 1")

## O projeto

## Cenário

O CEO da TechStore, Marcos, pediu um dashboard de performance da equipe comercial para a reunião de diretoria. Carlos delega para você:

> "Marcos quer ver **cada vendedor**, seu desempenho, os clientes que atende, os produtos que mais vende. Precisa estar tudo num relatório único e coeso. Você tem até amanhã."

Este projeto usa **todas as tabelas** do banco e combina JOINs, agregações e filtros avançados. Escreva cada consulta na caixa, execute e interprete.

---

## 1. Ranking Geral de Vendedores

**Pergunta de negócio:** Quem são os melhores vendedores e quanto atingiram da meta?

Para cada vendedor: posição no ranking, nome, região, total de pedidos entregues, faturamento, ticket médio, meta mensal e percentual de atingimento.

> 🎯 **Sua consulta:**
>

In [ ]:
# TODO: escreva a consulta e rode a célula
consulta = '''-- Escreva sua query aqui
'''
sql(consulta)

<details>
<summary>👀 Ver solução de referência</summary>

```sql
SELECT
  RANK() OVER (ORDER BY SUM(p.valor_total) DESC) AS posicao,
  v.nome AS vendedor,
  v.regiao,
  COUNT(p.id) AS total_pedidos,
  SUM(p.valor_total) AS faturamento,
  ROUND(AVG(p.valor_total), 2) AS ticket_medio,
  v.meta_mensal,
  ROUND(100.0 * SUM(p.valor_total) / v.meta_mensal, 1) AS pct_meta
FROM vendedores v
LEFT JOIN pedidos p
  ON p.vendedor_id = v.id
  AND p.status = 'entregue'
GROUP BY v.id, v.nome, v.regiao, v.meta_mensal
ORDER BY faturamento DESC;
```

</details>

**Resultado esperado (formato):**

| posicao | vendedor | regiao | total_pedidos | faturamento | ticket_medio | meta_mensal | pct_meta |
|---------|----------|--------|---------------|-------------|--------------|-------------|----------|
| 1 | Ricardo Mendes | Sudeste | 31 | 82695.30 | 2667.59 | 60000.00 | 137.8 |
| 2 | Juliana Rocha | Nordeste | 22 | 63078.00 | 2867.18 | 40000.00 | 157.7 |
| 3 | Fernanda Lima | Sul | 21 | 61382.60 | 2922.98 | 45000.00 | 136.4 |
| 4 | Carlos Eduardo | Sudeste | 19 | 53233.70 | 2801.77 | 55000.00 | 96.8 |
| 5 | Marcos Santana | Nordeste | 20 | 51217.90 | 2560.90 | 35000.00 | 146.3 |
| 6 | Bruno Alves | Centro-Oeste | 8 | 14682.90 | 1835.36 | 35000.00 | 42.0 |

---

## 2. Categorias por Vendedor (Top 3)

**Pergunta de negócio:** Quais categorias cada vendedor mais vende?

Para os 3 melhores vendedores (por faturamento), mostre as categorias que vendem, unidades e faturamento por categoria.

> 🎯 **Sua consulta:**
>

In [ ]:
# TODO: escreva a consulta e rode a célula
consulta = '''-- Escreva sua query aqui
'''
sql(consulta)

<details>
<summary>👀 Ver solução de referência</summary>

```sql
WITH top3 AS (
  SELECT v.id, v.nome
  FROM vendedores v
  JOIN pedidos p ON p.vendedor_id = v.id AND p.status = 'entregue'
  GROUP BY v.id, v.nome
  ORDER BY SUM(p.valor_total) DESC
  LIMIT 3
)
SELECT
  v.nome AS vendedor,
  pr.categoria,
  SUM(ip.quantidade) AS unidades,
  SUM(ip.quantidade * ip.preco_unitario) AS faturamento
FROM top3 v
JOIN pedidos p ON p.vendedor_id = v.id AND p.status = 'entregue'
JOIN itens_pedido ip ON ip.pedido_id = p.id
JOIN produtos pr ON pr.id = ip.produto_id
GROUP BY v.nome, pr.categoria
ORDER BY v.nome, faturamento DESC;
```

</details>

**Resultado esperado (formato):**

| vendedor | categoria | unidades | faturamento |
|----------|-----------|----------|-------------|
| Fernanda Lima | Notebooks | 8 | 32195.60 |
| Fernanda Lima | Smartphones | 9 | 15891.00 |
| Juliana Rocha | Notebooks | 4 | 24396.00 |
| Juliana Rocha | Smartphones | 11 | 22889.00 |
| Ricardo Mendes | Notebooks | 8 | 39094.70 |
| Ricardo Mendes | Smartphones | 11 | 16889.00 |

---

## 3. Clientes Atendidos por Vendedor

**Pergunta de negócio:** Quem são os 5 maiores clientes de cada vendedor?

Para cada vendedor, liste os 5 maiores clientes (por valor gasto). Mostre: vendedor, cliente, segmento, total gasto e qtd de pedidos.

> 🎯 **Sua consulta:**
>

In [ ]:
# TODO: escreva a consulta e rode a célula
consulta = '''-- Escreva sua query aqui
'''
sql(consulta)

<details>
<summary>👀 Ver solução de referência</summary>

```sql
WITH ranking_clientes AS (
  SELECT
    v.nome AS vendedor,
    c.nome AS cliente,
    c.segmento,
    SUM(p.valor_total) AS total_gasto,
    COUNT(p.id) AS qtd_pedidos,
    ROW_NUMBER() OVER (
      PARTITION BY v.id
      ORDER BY SUM(p.valor_total) DESC
    ) AS rn
  FROM pedidos p
  JOIN vendedores v ON v.id = p.vendedor_id
  JOIN clientes c ON c.id = p.cliente_id
  WHERE p.status = 'entregue'
  GROUP BY v.id, v.nome, c.id, c.nome, c.segmento
)
SELECT vendedor, cliente, segmento, total_gasto, qtd_pedidos
FROM ranking_clientes
WHERE rn <= 5
ORDER BY vendedor, total_gasto DESC;
```

</details>

**Resultado esperado (formato):**

| vendedor | cliente | segmento | total_gasto | qtd_pedidos |
|----------|---------|----------|-------------|-------------|
| Bruno Alves | Thiago Rocha | varejo | 7596.00 | 4 |
| Bruno Alves | Camila Santos | varejo | 7086.90 | 4 |
| Carlos Eduardo | InfoTech Solutions | corporativo | 15798.00 | 2 |
| Carlos Eduardo | Rafael Mendes | varejo | 7898.90 | 2 |
| Fernanda Lima | Tech Solutions LTDA | corporativo | 22196.80 | 5 |
| ... | ... | ... | ... | ... |

---

## 4. Taxa de Cancelamento por Vendedor

**Pergunta de negócio:** Quem está perdendo mais vendas por cancelamento?

Calcule por vendedor: total de pedidos, cancelados, taxa de cancelamento e valor perdido em cancelamentos.

> 🎯 **Sua consulta:**
>

In [ ]:
# TODO: escreva a consulta e rode a célula
consulta = '''-- Escreva sua query aqui
'''
sql(consulta)

<details>
<summary>👀 Ver solução de referência</summary>

```sql
SELECT
  v.nome AS vendedor,
  COUNT(p.id) AS total_pedidos,
  SUM(CASE WHEN p.status = 'cancelado' THEN 1 ELSE 0 END) AS cancelados,
  ROUND(
    100.0 * SUM(CASE WHEN p.status = 'cancelado' THEN 1 ELSE 0 END) / COUNT(p.id),
    1
  ) AS taxa_cancel,
  SUM(CASE WHEN p.status = 'cancelado' THEN p.valor_total ELSE 0 END) AS valor_perdido
FROM vendedores v
JOIN pedidos p ON p.vendedor_id = v.id
GROUP BY v.id, v.nome
HAVING ROUND(100.0 * SUM(CASE WHEN p.status = 'cancelado' THEN 1 ELSE 0 END) / COUNT(p.id), 1) > 10
ORDER BY taxa_cancel DESC;
```

</details>

**Resultado esperado (formato):**

| vendedor | total_pedidos | cancelados | taxa_cancel | valor_perdido |
|----------|---------------|------------|-------------|---------------|
| Bruno Alves | 14 | 2 | 14.3 | 4482.10 |
| Carlos Eduardo | 28 | 3 | 10.7 | 13378.00 |

> 💡 **Interprete:** Taxa acima de 10% é zona vermelha em vendas B2C. Vale investigar, é o vendedor, o cliente ou a forma de pagamento?

---

## 5. Produtos Campeões por Região

**Pergunta de negócio:** Em cada região, quais são os 3 produtos que mais vendem?

Use Window Function (ROW_NUMBER OVER PARTITION BY regiao) para pegar os top 3 de cada região.

> 🎯 **Sua consulta:**
>

In [ ]:
# TODO: escreva a consulta e rode a célula
consulta = '''-- Escreva sua query aqui
'''
sql(consulta)

<details>
<summary>👀 Ver solução de referência</summary>

```sql
WITH ranking AS (
  SELECT
    v.regiao,
    pr.nome AS produto,
    pr.categoria,
    SUM(ip.quantidade) AS unidades,
    SUM(ip.quantidade * ip.preco_unitario) AS faturamento,
    ROW_NUMBER() OVER (
      PARTITION BY v.regiao
      ORDER BY SUM(ip.quantidade * ip.preco_unitario) DESC
    ) AS rn
  FROM itens_pedido ip
  JOIN pedidos p ON p.id = ip.pedido_id
  JOIN produtos pr ON pr.id = ip.produto_id
  JOIN vendedores v ON v.id = p.vendedor_id
  WHERE p.status = 'entregue'
  GROUP BY v.regiao, pr.id, pr.nome, pr.categoria
)
SELECT regiao, produto, categoria, unidades, faturamento
FROM ranking
WHERE rn <= 3
ORDER BY regiao, faturamento DESC;
```

</details>

**Resultado esperado (formato):**

| regiao | produto | categoria | unidades | faturamento |
|--------|---------|-----------|----------|-------------|
| Sudeste | Notebook Gamer Z5 | Notebooks | 4 | 31596.00 |
| Sudeste | Notebook Pro X1 | Notebooks | 4 | 18399.60 |
| Sudeste | Monitor UltraWide 34" | Monitores | 4 | 11596.00 |
| Nordeste | Smartphone Galaxy Ultra | Smartphones | 7 | 23093.00 |
| Nordeste | MacStation Desktop | Notebooks | 4 | 17196.00 |
| Sul | Notebook Pro X1 | Notebooks | 4 | 18399.60 |

---

## 6. Análise Temporal, Faturamento Mensal por Vendedor

**Pergunta de negócio:** Como cada vendedor evoluiu nos últimos 3 meses?

Mostre o faturamento mensal de cada vendedor nos meses de jan/fev/mar de 2024.

> 🎯 **Sua consulta:**
>

In [ ]:
# TODO: escreva a consulta e rode a célula
consulta = '''-- Escreva sua query aqui
'''
sql(consulta)

<details>
<summary>👀 Ver solução de referência</summary>

```sql
SELECT
  v.nome AS vendedor,
  SUM(CASE WHEN EXTRACT(MONTH FROM p.data_pedido) = 1 THEN p.valor_total ELSE 0 END) AS jan_2024,
  SUM(CASE WHEN EXTRACT(MONTH FROM p.data_pedido) = 2 THEN p.valor_total ELSE 0 END) AS fev_2024,
  SUM(CASE WHEN EXTRACT(MONTH FROM p.data_pedido) = 3 THEN p.valor_total ELSE 0 END) AS mar_2024
FROM vendedores v
LEFT JOIN pedidos p
  ON p.vendedor_id = v.id
  AND p.status = 'entregue'
  AND EXTRACT(YEAR FROM p.data_pedido) = 2024
GROUP BY v.id, v.nome
ORDER BY mar_2024 DESC;
```

</details>

**Resultado esperado (formato):**

| vendedor | jan_2024 | fev_2024 | mar_2024 |
|----------|----------|----------|----------|
| Fernanda Lima | 12796.90 | 14894.90 | 24994.80 |
| Carlos Eduardo | 2698.00 | 20544.90 | 19195.80 |
| Juliana Rocha | 10396.00 | 17195.00 | 16192.00 |
| Ricardo Mendes | 16684.90 | 30442.90 | 14374.60 |
| Marcos Santana | 17393.90 | 17030.00 | 12495.00 |

> 💡 **Interprete:** Quem está crescendo? Quem caiu? Quedas sucessivas pedem conversa.

---

## 7. Alerta: Vendedores Abaixo de 80% da Meta

**Pergunta de negócio:** Quem precisa de plano de ação agora?

Liste os vendedores que faturaram menos de 80% da meta mensal.

> 🎯 **Sua consulta:**
>

In [ ]:
# TODO: escreva a consulta e rode a célula
consulta = '''-- Escreva sua query aqui
'''
sql(consulta)

<details>
<summary>👀 Ver solução de referência</summary>

```sql
SELECT
  v.nome AS vendedor,
  v.regiao,
  SUM(p.valor_total) AS faturamento,
  v.meta_mensal,
  ROUND(100.0 * SUM(p.valor_total) / v.meta_mensal, 1) AS pct_meta
FROM vendedores v
LEFT JOIN pedidos p
  ON p.vendedor_id = v.id
  AND p.status = 'entregue'
GROUP BY v.id, v.nome, v.regiao, v.meta_mensal
HAVING ROUND(100.0 * SUM(p.valor_total) / v.meta_mensal, 1) < 80
ORDER BY pct_meta ASC;
```

</details>

---

## ✅ Checklist de Entrega

- [ ] As 7 consultas executam sem erro
- [ ] JOINs conectam as tabelas corretas (vendedores, pedidos, itens_pedido, produtos, clientes)
- [ ] LEFT JOIN usado quando preciso considerar vendedores sem vendas
- [ ] Window Functions (ROW_NUMBER, RANK, PARTITION BY) usadas onde indicado
- [ ] GROUP BY + HAVING aplicados corretamente
- [ ] Interpretações agregam valor além dos números

> 💡 **Reflexão:** Esse é o tipo de relatório que vai pra reunião de diretoria. A clareza na estrutura é tão importante quanto a precisão dos dados.

## Do notebook ao portfólio

Antes de publicar no GitHub:

- [ ] O notebook roda do início ao fim num ambiente limpo (Kernel > Restart & Run All).
- [ ] As células TODO estão resolvidas e os resultados aparecem nas saídas.
- [ ] Nenhuma chave, senha ou token no código: tudo vem de variável de ambiente, e o `.gitignore` tem `.env`.
- [ ] O README abre com o problema e o resultado, não com a tecnologia.
- [ ] Um gráfico ou uma tabela de métricas no README (recrutador lê em 30 segundos).
- [ ] Commits pequenos, com mensagens claras (`feat:`, `fix:`, `docs:`).

O que destacar: a pergunta de negócio, a decisão que você tomou com os dados, como você mediu se funcionou (métricas, testes, avaliação) e o que faria diferente em produção. Diga que os dados são fictícios e que o projeto nasceu num curso; o que conta é o seu raciocínio.

## README sugerido

A célula abaixo grava `README.md`, `requirements.txt` e `.gitignore` na pasta de trabalho. Troque os TODO pelos seus resultados antes de publicar.

# Dashboard de Performance da Equipe

O CEO da TechStore, Marcos, pediu um dashboard de performance da equipe comercial para a reunião de diretoria. Carlos delega para você:

Projeto desenvolvido na formação Analista de Dados, Cientista de Dados, Engenheiro de Dados da DataMundo, no percurso "SQL para Análise de Dados".

## O que este projeto faz

- TODO: liste aqui as partes do projeto

## Tecnologias

- Python, duckdb, pandas

## Como rodar

```bash
python -m venv .venv
source .venv/bin/activate  # no Windows: .venv\Scripts\activate
pip install -r requirements.txt jupyter
jupyter notebook projeto-dashboard-de-performance-da-equipe.ipynb
```

## Dados

Banco fictício da TechStore, criado pelo próprio notebook num DuckDB local.

## Resultados

TODO: os números principais, uma tabela de métricas ou um gráfico (salve a imagem em `docs/` e mostre aqui).

## Decisões e aprendizados

TODO: duas ou três decisões que você tomou, o que testou e o que mudaria.

## Próximos passos

TODO: o que faria com mais tempo ou com dados reais.

In [ ]:
from pathlib import Path

Path("README.md").write_text('''# Dashboard de Performance da Equipe

O CEO da TechStore, Marcos, pediu um dashboard de performance da equipe comercial para a reunião de diretoria. Carlos delega para você:

Projeto desenvolvido na formação Analista de Dados, Cientista de Dados, Engenheiro de Dados da DataMundo, no percurso "SQL para Análise de Dados".

## O que este projeto faz

- TODO: liste aqui as partes do projeto

## Tecnologias

- Python, duckdb, pandas

## Como rodar

```bash
python -m venv .venv
source .venv/bin/activate  # no Windows: .venv\\Scripts\\activate
pip install -r requirements.txt jupyter
jupyter notebook projeto-dashboard-de-performance-da-equipe.ipynb
```

## Dados

Banco fictício da TechStore, criado pelo próprio notebook num DuckDB local.

## Resultados

TODO: os números principais, uma tabela de métricas ou um gráfico (salve a imagem em `docs/` e mostre aqui).

## Decisões e aprendizados

TODO: duas ou três decisões que você tomou, o que testou e o que mudaria.

## Próximos passos

TODO: o que faria com mais tempo ou com dados reais.
''', encoding="utf-8")
Path("requirements.txt").write_text('''duckdb
pandas
''', encoding="utf-8")
Path(".gitignore").write_text(".env\n.venv/\n__pycache__/\n.ipynb_checkpoints/\n", encoding="utf-8")
print("README.md, requirements.txt e .gitignore gravados em", Path.cwd())